#### Boston Dataset - Crime Rate Prediction

Some notes on this question:

- In my solutions I went and computed dummy variables for `chas` and `rad` considering they are categorical variables. In answers I've looked at online they simply used them as-is. I also recognise that perhaps there's no need to one-hot encode `rad` because it is an ordinal categorical variable, which means that there is a difference between 1 and 2, and this is somewhat well-represented by the ordinal values.
- Obviously, if I remove this one-hot preprocessing the results would be quite different. But I think there's no need to spend more time on this question rectifying the results as the focus is on exploring univariate / multiple / polynomial regression which I have thoroughly done.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from ISLP import load_data

Boston = load_data("Boston")
# Boston.dtypes
Boston.head()
# response: crim (Crime Rate)
# predictors: all other variables

In [ ]:
Boston.nunique()
# chas and rad are categorical variables

In [ ]:
# check null values
Boston[Boston.isnull().any(axis=1)]

(a) For each predictor, fit a simple linear regression model to predict the response (crim)
- I observed the F-statistic p-value to check if there is a relationship at all, and the individual t-statistic p-values for each predictor.
- `chas` failed the F-test, this indicates that there is no evidence of a relationship between `crim` and `chas`.
- For the values in `rad`, only `rad_24` showed a significant t-statistic p-value, with the other categories showing high p-values. Hence, for the `rad` predictor, we can conclude that there is only a statistically significant association between it being =24 and `crim`, in the presence of other `rad` predictors.
- For all other predictors, they showed statistically significant associations, measured at 95% level of significance.

In [ ]:
y = Boston['crim']
predictors = Boston.drop(columns=['crim'])
intercept = pd.DataFrame({
    "intercept": np.ones(predictors.shape[0])
})
coefficients_indiv = [] # for later, part (c)
for col in predictors.columns:
    if col in ['chas', 'rad']:
        X = pd.concat([intercept, pd.get_dummies(Boston[[col]], columns=[col], drop_first=True, dtype=int)], axis=1)
    else:
        X = pd.concat([intercept, Boston[[col]]], axis=1)
    print(X.columns)

    result = sm.OLS(y, X).fit()
    print(f"{col} F-pvalue: {result.f_pvalue}")
    print(pd.DataFrame({
        "coefficient": result.params,
        "p-value": result.pvalues,
    }, index=result.params.index))
    print("-----")

    coefficients_indiv.append(result.params.loc[result.params.index.drop("intercept")])
coefficients_indiv = pd.concat(coefficients_indiv)
# print(coefficients_indiv)

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(Boston['chas'], y) # for 'chas', there is no statistically significant association. we can see this from the scatter plot

(b) Fit a linear model using all predictors
- From the results (t-statistic p-values), we can reject the null hypothesis $H_0:\beta_j=0$ for the predictors `zn`, `dis`, `medv`, `rad_24` at 95% level of significance.

In [ ]:
X = pd.get_dummies(predictors, columns=['chas', 'rad'], drop_first=True, dtype=int)
X = pd.concat([intercept, X], axis=1)
result = sm.OLS(y, X).fit()

coefficients_together = result.params.loc[result.params.index.drop("intercept")].to_frame()

result.summary()

(c) Comparison of results in (a) and in (b)
- When the predictors are fitted to the response individually, `zn`, `indus`, `nox`, `rm`, `age`, `dis`, `rad_24`, `tax`, `ptratio`, `lstat`, `medv` all showed a statistically significant relationship. However, when fitted to the response together, only `zn`, `dis`, `medv`, `rad_24` showed a statistically significant relationship. It is possible that there is some collinearity present in the data, as such `zn`, `dis`, `medv` and `rad_24` were able to mostly explain the patterns that the other variables individually explained. 
- From the plot of multiple regression coefficients against univariate regression coefficients, most of the points are centered around (0,0) and are close to each other, showing small variation in coefficient values when changing from univariate to multiple regression. There are two coefficients that do not follow this trend, namely `nox` and `rad_24`.

In [ ]:
coefficients = pd.concat([coefficients_indiv, coefficients_together], axis=1)
coefficients.columns = ["indiv", "together"]
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(coefficients["indiv"], coefficients["together"])
ax.set_xlabel("univariate regression coefficients")
ax.set_ylabel("multiple regression coefficients")
# most of the points are close to each other, except rad_24 and nox
ax.annotate("nox", (30, -10))
ax.annotate("rad_24", (13.5, 12.7))
plt.tight_layout()

(d) Checking for Non-linear relationships by modelling for each variable, $Y=\beta_0+\beta_1X+\beta_2X^2+\beta_3X^3+\epsilon$
- Categorical variables `chas` and `rad` were not included in this analysis.
- `zn`: Does not show evidence for non-linear association based on p-values.
- `indus`: Shows evidence for non-linear association based on p-values, up to $X^3$
- `nox`: Shows evidence for non-linear association based on p-values, up to $X^3$
- `rm`: Does not show evidence for non-linear association
- `age`: Shows evidence for non-linear association based on p-values, up to $X^3$
- `dis`: Shows evidence for non-linear association based on p-values, up to $X^3$
- `tax`: Does not show evidence for non-linear association
- `ptratio`: Shows evidence for non-linear association based on p-values, up to $X^3$
- `lstat`: Does not show evidence for non-linear association
- `medv`: Shows evidence for non-linear association based on p-values, up to $X^3$

In [ ]:
for col in predictors.columns:
    if col in ['chas', 'rad']:
        continue
    
    X = pd.DataFrame({
        "intercept": np.ones(Boston.shape[0]),
        f"{col}": Boston[col],
        f"{col}^2": np.pow(Boston[col], 2),
        f"{col}^3": np.pow(Boston[col], 3),
    })

    result = sm.OLS(y, X).fit()
    print(f"{col} F-pvalue: {result.f_pvalue}")
    print(pd.DataFrame({
        "coefficient": result.params,
        "p-value": result.pvalues,
    }, index=result.params.index))
    print("-----")